# Parcial 1 — Programación para Ciencia de Datos (SCY1101)

**Dataset:** `urgencias_noprocesados_grupo06.csv`  
**Objetivo:** Resolver el encargo completo de la rúbrica: manipulación, limpieza, transformaciones avanzadas, validación e informe con visualizaciones.

## 1) Plan de trabajo alineado a la rúbrica (Encargo)

1. **Manipulación en Pandas:** filtros, agrupaciones múltiples y joins.
2. **Reporte y visualización:** métricas + gráficos claros.
3. **Transformaciones avanzadas:** vectorización, broadcasting, pivot/reshape, chunking.
4. **Flujo de limpieza:** estandarización, imputación, deduplicación y consistencia.
5. **Entorno reproducible:** evidencia de versión de librerías y `requirements.txt`.
6. **Integridad/procedencia:** hash de archivo, validación de esquema y reglas de calidad.

In [1]:
# Importa comportamiento futuro para anotaciones de tipos más limpias.
from __future__ import annotations

# Librería para calcular hash SHA256 (integridad/procedencia de archivos).
import hashlib
# Manejo de rutas de forma segura y multiplataforma.
from pathlib import Path
# Permite parsear fechas con formatos definidos.
from datetime import datetime
# Para silenciar warnings no críticos en la salida.
import warnings

# NumPy: operaciones numéricas eficientes.
import numpy as np
# Pandas: manipulación tabular de datos.
import pandas as pd
# Matplotlib: generación de gráficos.
import matplotlib.pyplot as plt

# Silencia warnings para que el notebook sea más limpio visualmente.
warnings.filterwarnings("ignore")
# Muestra todas las columnas al imprimir DataFrames.
pd.set_option("display.max_columns", None)

# Función que detecta automáticamente la raíz del proyecto.
def detect_project_root() -> Path:
    """Detecta raíz del proyecto aunque el notebook se ejecute desde /notebooks."""
    # Prueba primero el directorio actual y luego su padre.
    candidates = [Path.cwd(), Path.cwd().parent]
    # Recorre candidatos y valida presencia de archivo de datos + rúbrica.
    for base in candidates:
        # Acepta CSV en raíz o dentro de data/raw (estructura organizada).
        has_csv = (base / 'urgencias_noprocesados_grupo06.csv').exists() or (base / 'data' / 'raw' / 'urgencias_noprocesados_grupo06.csv').exists()
        # Acepta PDF en raíz original o dentro de docs.
        has_pdf = (base / 'EV PARCIAL 1 SCY1101_ESTUDIANTE.pdf').exists() or (base / 'docs' / 'rubrica_parcial1.pdf').exists()
        if has_csv and has_pdf:
            # Devuelve la ruta absoluta detectada como raíz.
            return base.resolve()
    # Si no encuentra archivos ancla, detiene con error explícito.
    raise FileNotFoundError('No se encontró la raíz del proyecto (faltan CSV o PDF).')

# Guarda la ruta raíz detectada.
PROJECT_ROOT = detect_project_root()
# Define carpeta para datos.
DATA_DIR = PROJECT_ROOT / 'data'
# Define carpeta para salidas (gráficos/reportes).
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
# Define carpeta de notebooks.
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

# Crea carpetas si no existen.
for directory in [DATA_DIR, OUTPUTS_DIR, NOTEBOOKS_DIR]:
    directory.mkdir(exist_ok=True)

# Posibles ubicaciones del CSV bruto (orden de prioridad).
raw_candidates = [
    PROJECT_ROOT / 'urgencias_noprocesados_grupo06.csv',
    DATA_DIR / 'raw' / 'urgencias_noprocesados_grupo06.csv',
    DATA_DIR / 'urgencias_noprocesados_grupo06.csv',
]
# Selecciona la primera ruta existente; si ninguna existe, devuelve None.
RAW_PATH = next((p for p in raw_candidates if p.exists()), None)
# Valida que se haya encontrado el archivo fuente.
if RAW_PATH is None:
    raise FileNotFoundError('No se encontró urgencias_noprocesados_grupo06.csv en rutas esperadas.')

# Ruta destino del dataset procesado.
PROCESSED_PATH = DATA_DIR / 'urgencias_procesados_grupo06.csv'

# Muestra info de contexto para depuración/reproducibilidad.
print('Directorio de trabajo actual:', Path.cwd().resolve())
print('PROJECT_ROOT detectado:', PROJECT_ROOT)
print('Archivo fuente:', RAW_PATH)
print('Archivo fuente existe:', RAW_PATH.exists())


Directorio de trabajo actual: /Users/benja/Documents/Duoc/5to Semestre/Programacion Ciencia de Datos/EA1/Parcial 1/notebooks
PROJECT_ROOT detectado: /Users/benja/Documents/Duoc/5to Semestre/Programacion Ciencia de Datos/EA1/Parcial 1
Archivo fuente: /Users/benja/Documents/Duoc/5to Semestre/Programacion Ciencia de Datos/EA1/Parcial 1/urgencias_noprocesados_grupo06.csv
Archivo fuente existe: True


In [2]:
# Calcula SHA256 de un archivo para demostrar integridad/procedencia.
def sha256_file(path: Path) -> str:
    """Calcula hash SHA256 de un archivo para control de procedencia."""
    # Inicializa objeto hash SHA256.
    h = hashlib.sha256()
    # Abre archivo en modo binario para leer bytes reales.
    with open(path, 'rb') as f:
        # Lee en bloques de 8192 bytes para eficiencia de memoria.
        for chunk in iter(lambda: f.read(8192), b''):
            # Actualiza hash con cada bloque leído.
            h.update(chunk)
    # Devuelve hash final en formato hexadecimal.
    return h.hexdigest()


# Convierte textos de fecha con formatos mixtos a Timestamp de pandas.
def parse_mixed_date(value: str):
    """Parsea fechas en múltiples formatos comunes del dataset."""
    # Si viene nulo, retorna NaT (fecha nula en pandas).
    if pd.isna(value):
        return pd.NaT

    # Limpia espacios sobrantes del texto.
    value = str(value).strip()
    # Formatos esperados en el dataset.
    fmts = ["%d.%m.%y", "%d-%m-%Y", "%Y/%m/%d", "%d.%m.%Y", "%d-%m-%y"]

    # Intenta parsear usando cada formato conocido.
    for fmt in fmts:
        try:
            # Si parsea, devuelve fecha normalizada como Timestamp.
            return pd.Timestamp(datetime.strptime(value, fmt).date())
        except ValueError:
            # Si falla formato actual, sigue al siguiente.
            continue

    # Fallback robusto: intenta parseo general con dayfirst=True.
    return pd.to_datetime(value, errors='coerce', dayfirst=True)


# Devuelve la moda de una serie (valor más frecuente) o NaN si no hay datos.
def mode_or_nan(series: pd.Series):
    """Retorna la moda de una serie o NaN si está vacía."""
    # mode() puede devolver múltiples valores; tomamos el primero.
    m = series.mode(dropna=True)
    # Si hay moda devuelve primer valor, si no, NaN.
    return m.iloc[0] if not m.empty else np.nan


In [3]:
# Calcula hash del archivo crudo para trazabilidad.
raw_hash = sha256_file(RAW_PATH)

# Intenta leer el CSV original.
try:
    df_raw = pd.read_csv(RAW_PATH)
except Exception as e:
    # Si falla lectura, detiene con mensaje claro.
    raise RuntimeError(f"No se pudo leer el CSV: {e}")

# Muestra tamaño de tabla cargada: (filas, columnas).
print(f"Filas x columnas: {df_raw.shape}")
# Muestra hash del archivo original.
print(f"SHA256 archivo original: {raw_hash}")
# Vista rápida de primeras filas para inspección inicial.
df_raw.head(3)


Filas x columnas: (4742, 29)
SHA256 archivo original: 9e149a01372933d09829bab46bdce7418c009e518b2d3d887efdbd08b297555f


,EstablecimientoCodigo,EstablecimientoGlosa,RegionCodigo,RegionGlosa,ComunaCodigo,ComunaGlosa,ServicioSaludCodigo,ServicioSaludGlosa,TipoEstablecimiento,DependenciaAdministrativa,NivelAtencion,TipoUrgencia,Latitud,Longitud,NivelComplejidad,Anio,SemanaEstadistica,OrdenCausa,Causa,NumTotal,NumMenor1Anio,Num1a4Anios,Num5a14Anios,Num15a64Anios,Num65oMas,FechaAtencionTexto,SexoPaciente,PrioridadTriage,CostoAtencionCLP
0,110901,Hospital Regional de Arica Dr. Juan Noe,15,Arica y Parinacota,15101,Arica,1,Servicio de Salud Arica,Hospital,Servicio de Salud,Terciario,Hospitalaria (UEH),-18.4783,-70.3126,Alta Complejidad,2023,1,1,Influenza y neumonia,201,19,21,27,87,47,02.01.23,F,C5,87482.0
1,110901,Hospital Regional de Arica Dr. Juan Noe,15,Arica y Parinacota,15101,Arica,1,Servicio de Salud Arica,Hospital,Servicio de Salud,Terciario,Hospitalaria (UEH),-18.4783,-70.3126,Alta Complejidad,2023,1,2,Infeccion respiratoria aguda alta,193,14,25,30,106,18,02.01.23,F,C4,105598.0
2,110901,Hospital Regional de Arica Dr. Juan Noe,15,Arica y Parinacota,15101,Arica,1,Servicio de Salud Arica,Hospital,Servicio de Salud,Terciario,Hospitalaria (UEH),-18.4783,-70.3126,Alta Complejidad,2023,1,3,Bronquitis/Bronquiolitis aguda,94,18,19,23,588,6,02-01-2023,F,C2,135016.0


In [4]:
# Imprime tipo de dato por columna (int, float, string, etc.).
print('Tipos de datos:')
print(df_raw.dtypes)
print()
# Muestra top de columnas con más nulos para priorizar limpieza.
print('Nulos por columna (top 15):')
print(df_raw.isna().sum().sort_values(ascending=False).head(15))
print()
# Cuenta duplicados exactos de filas completas.
print('Duplicados exactos:', df_raw.duplicated().sum())

# Lista de columnas por tramos de edad.
age_cols = ['NumMenor1Anio', 'Num1a4Anios', 'Num5a14Anios', 'Num15a64Anios', 'Num65oMas']
# Suma por fila de todos los tramos etarios.
sum_age = df_raw[age_cols].sum(axis=1)
# Cuenta cuántas filas tienen NumTotal inconsistente con suma por tramos.
print('Registros donde NumTotal != suma por tramos etarios:', (df_raw['NumTotal'] != sum_age).sum())


Tipos de datos:
EstablecimientoCodigo          int64
EstablecimientoGlosa             str
RegionCodigo                   int64
RegionGlosa                      str
ComunaCodigo                   int64
ComunaGlosa                      str
ServicioSaludCodigo            int64
ServicioSaludGlosa               str
TipoEstablecimiento              str
DependenciaAdministrativa        str
NivelAtencion                    str
TipoUrgencia                     str
Latitud                      float64
Longitud                     float64
NivelComplejidad                 str
Anio                           int64
SemanaEstadistica              int64
OrdenCausa                     int64
Causa                            str
NumTotal                       int64
NumMenor1Anio                  int64
Num1a4Anios                    int64
Num5a14Anios                   int64
Num15a64Anios                  int64
Num65oMas                      int64
FechaAtencionTexto               str
SexoPaciente          

## 2) Limpieza de datos con justificación técnica

Decisiones aplicadas:

- **Deduplicación**: se eliminan duplicados exactos para evitar doble conteo.
- **Estandarización de categorías**: `SexoPaciente` y `PrioridadTriage` tenían formatos inconsistentes.
- **Fechas mixtas**: normalización a `datetime` con parser multi-formato.
- **Imputación dirigida**:
  - Coordenadas por mediana del establecimiento.
  - Variables categóricas por moda del establecimiento.
  - Costo por mediana de prioridad de triage (fallback global).
- **Consistencia interna**: se recalcula `NumTotal` como suma de tramos etarios para eliminar contradicciones.
- **Outliers de costo**: winsorización p1-p99 para reducir impacto de extremos sin borrar casos.

In [ ]:
# Crea copia de trabajo para no modificar df_raw directamente.
df = df_raw.copy()

# Elimina duplicados exactos para evitar doble conteo en análisis.
df = df.drop_duplicates().copy()

# Columnas de texto a normalizar (quitar espacios al inicio/final).
text_cols = [
    'EstablecimientoGlosa', 'RegionGlosa', 'ComunaGlosa', 'ServicioSaludGlosa',
    'TipoEstablecimiento', 'DependenciaAdministrativa', 'NivelAtencion',
    'TipoUrgencia', 'NivelComplejidad', 'Causa', 'SexoPaciente', 'PrioridadTriage'
]
# Recorre cada columna de texto y aplica limpieza básica.
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype('string').str.strip()

# Mapa de normalización para sexo (distintas variantes -> estándar).
sexo_map = {
    'F': 'F', 'f': 'F', 'femenino': 'F', 'Femenino': 'F',
    'M': 'M', 'm': 'M', 'masculino': 'M', 'Masculino': 'M'
}
# Reemplaza variantes por valores normalizados.
df['SexoPaciente'] = df['SexoPaciente'].replace(sexo_map)
# Si no es F/M, se marca como ND (no disponible).
df['SexoPaciente'] = df['SexoPaciente'].where(df['SexoPaciente'].isin(['F', 'M']), 'ND')

# Limpia texto de triage: mayúsculas, sin espacios, sin guiones.
triage_clean = (
    df['PrioridadTriage']
    .astype('string')
    .str.upper()
    .str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)
)
# Si no es C1..C5, se marca ND.
triage_clean = triage_clean.where(triage_clean.isin(['C1', 'C2', 'C3', 'C4', 'C5']), 'ND')
# Guarda triage ya normalizado.
df['PrioridadTriage'] = triage_clean

# Convierte texto de fecha a tipo fecha (Timestamp).
df['FechaAtencion'] = df['FechaAtencionTexto'].apply(parse_mixed_date)

# Imputa coordenadas faltantes por mediana del establecimiento.
for col in ['Latitud', 'Longitud']:
    # Mediana por establecimiento para esa columna.
    grp_med = df.groupby('EstablecimientoCodigo')[col].transform('median')
    # Primero rellena con mediana local (más coherente).
    df[col] = df[col].fillna(grp_med)
    # Si aún faltan, rellena con mediana global.
    df[col] = df[col].fillna(df[col].median())

# Imputa categóricas faltantes usando moda por establecimiento.
for col in ['DependenciaAdministrativa', 'TipoUrgencia', 'NivelComplejidad']:
    # Moda por establecimiento.
    fill_by_est = df.groupby('EstablecimientoCodigo')[col].transform(mode_or_nan)
    # Completa nulos con moda local.
    df[col] = df[col].fillna(fill_by_est)
    # Si aún hay nulos, usa etiqueta genérica.
    df[col] = df[col].fillna('NoInformado')

# Mediana de costo por triage para imputar faltantes de costo.
cost_med_triage = df.groupby('PrioridadTriage')['CostoAtencionCLP'].transform('median')
# Rellena costo faltante con mediana del grupo de triage.
df['CostoAtencionCLP'] = df['CostoAtencionCLP'].fillna(cost_med_triage)
# Fallback final: mediana global.
df['CostoAtencionCLP'] = df['CostoAtencionCLP'].fillna(df['CostoAtencionCLP'].median())

# Guarda NumTotal original para auditoría de cambios.
df['NumTotalOriginal'] = df['NumTotal']
# Recalcula NumTotal desde tramos etarios (corrige inconsistencias).
df['NumTotal'] = df[age_cols].sum(axis=1)
# Marca filas donde se corrigió el total.
df['FlagTotalCorregido'] = df['NumTotalOriginal'] != df['NumTotal']

# Calcula percentiles 1 y 99 para controlar valores extremos de costo.
q1, q99 = df['CostoAtencionCLP'].quantile([0.01, 0.99])
# Winsorización: recorta outliers al rango [p1, p99].
df['CostoAtencionCLP'] = df['CostoAtencionCLP'].clip(lower=q1, upper=q99)

# Reporte rápido de estado post-limpieza.
print('Registros tras limpieza:', len(df))
print('Nulos remanentes top 10:')
print(df.isna().sum().sort_values(ascending=False).head(10))


## 3) Manipulación en Pandas (filtros, agrupaciones y joins)

In [ ]:
# Define condiciones de filtro analítico.
filtro = (
    # Años más recientes.
    (df['Anio'] >= 2024) &
    # Solo establecimientos hospitalarios.
    (df['TipoEstablecimiento'].str.contains('Hospital', case=False, na=False)) &
    # Triage de mayor prioridad analítica.
    (df['PrioridadTriage'].isin(['C1', 'C2', 'C3']))
)

# Aplica filtro y crea subset resultante.
df_filtro = df.loc[filtro].copy()
# Muestra tamaño del subconjunto filtrado.
print('Registros filtrados:', df_filtro.shape)


In [ ]:
# Agrupa por región, sexo y triage para construir resumen agregado.
agg_multi = (
    df_filtro
    .groupby(['RegionGlosa', 'SexoPaciente', 'PrioridadTriage'], as_index=False)
    .agg(
        # Suma de atenciones por grupo.
        atenciones_totales=('NumTotal', 'sum'),
        # Suma de costos por grupo.
        costo_total=('CostoAtencionCLP', 'sum'),
        # Costo promedio por grupo.
        costo_promedio=('CostoAtencionCLP', 'mean'),
        # Cantidad de filas por grupo.
        n_registros=('NumTotal', 'size')
    )
    # Ordena para priorizar grupos de mayor volumen/costo.
    .sort_values(['atenciones_totales', 'costo_total'], ascending=False)
)

# Muestra top 10 filas del resumen.
agg_multi.head(10)


In [ ]:
# Diccionario región -> macrozona para enriquecer análisis territorial.
macrozona_map = {
    'Arica y Parinacota': 'Norte',
    'Tarapaca': 'Norte',
    'Antofagasta': 'Norte',
    'Atacama': 'Norte',
    'Coquimbo': 'Norte Chico',
    'Valparaiso': 'Centro',
    'Metropolitana de Santiago': 'Centro',
    "Libertador General Bernardo O'Higgins": 'Centro Sur',
    'Maule': 'Centro Sur',
    'Nuble': 'Sur',
    'Biobio': 'Sur',
    'La Araucania': 'Sur',
    'Los Rios': 'Sur Austral',
    'Los Lagos': 'Sur Austral',
    'Aysen del General Carlos Ibanez del Campo': 'Austral',
    'Magallanes y de la Antartica Chilena': 'Austral'
}

# Construye dimensión de región sin duplicados y agrega macrozona.
dim_region = (
    df[['RegionCodigo', 'RegionGlosa']]
    .drop_duplicates()
    .assign(Macrozona=lambda x: x['RegionGlosa'].map(macrozona_map).fillna('NoClasificada'))
)

# Construye dimensión temporal (año/semana -> trimestre aproximado).
dim_calendar = (
    df[['Anio', 'SemanaEstadistica']]
    .drop_duplicates()
    .assign(
        TrimestreAprox=lambda x: pd.cut(
            # Semana estadística que se va a binnear.
            x['SemanaEstadistica'],
            # Límites de trimestre por semanas.
            bins=[0, 13, 26, 39, 53],
            # Etiquetas de salida.
            labels=['T1', 'T2', 'T3', 'T4'],
            # Incluye borde inferior en primer intervalo.
            include_lowest=True
        )
    )
)

# Enriquecimiento: merge con región + merge con calendario.
df_enriched = (
    df
    .merge(dim_region, on=['RegionCodigo', 'RegionGlosa'], how='left')
    .merge(dim_calendar, on=['Anio', 'SemanaEstadistica'], how='left')
)

# Vista de columnas enriquecidas.
df_enriched[['RegionGlosa', 'Macrozona', 'Anio', 'SemanaEstadistica', 'TrimestreAprox']].head()


## 4) Transformaciones avanzadas (vectorización, broadcasting, pivot, reshape, chunking)

In [ ]:
# Crea costo unitario por atención usando vectorización (sin loop explícito).
df_enriched['CostoUnitarioCLP'] = np.where(
    # Condición: evita división por cero.
    df_enriched['NumTotal'] > 0,
    # Valor si condición se cumple.
    df_enriched['CostoAtencionCLP'] / df_enriched['NumTotal'],
    # Valor si condición no se cumple.
    np.nan
)

# Convierte tramos etarios a matriz NumPy para operaciones numéricas masivas.
age_matrix = df_enriched[age_cols].to_numpy(dtype=float)
# Media por columna (cada tramo etario).
means = age_matrix.mean(axis=0)
# Desviación estándar por columna.
stds = age_matrix.std(axis=0)
# Evita división por cero en columnas con desvío 0.
stds = np.where(stds == 0, 1, stds)
# Broadcasting: z-score = (x - media) / std para cada columna.
age_z = (age_matrix - means) / stds

# Guarda z-score de cada tramo como nuevas columnas.
for i, col in enumerate(age_cols):
    df_enriched[f'{col}_z'] = age_z[:, i]

# Pivot table: región en filas, triage en columnas, suma de atenciones como valor.
pivot_region_triage = pd.pivot_table(
    df_enriched,
    index='RegionGlosa',
    columns='PrioridadTriage',
    values='NumTotal',
    aggfunc='sum',
    fill_value=0
)

# Reshape a formato largo para facilitar ciertos análisis/gráficos.
df_long_edades = df_enriched.melt(
    id_vars=['RegionGlosa', 'Anio', 'PrioridadTriage'],
    value_vars=age_cols,
    var_name='TramoEtario',
    value_name='Atenciones'
)

# Muestra dimensiones resultantes de pivot y reshape.
print('Pivot shape:', pivot_region_triage.shape)
print('Formato long shape:', df_long_edades.shape)


In [ ]:
# Lista para acumular resultados parciales por chunk.
chunk_agg = []
# Lee CSV por lotes de 1000 filas (útil en datasets grandes).
for chunk in pd.read_csv(RAW_PATH, chunksize=1000):
    # Resume cada chunk por región.
    tmp = chunk.groupby('RegionGlosa', as_index=False)['NumTotal'].sum()
    # Guarda resumen parcial.
    chunk_agg.append(tmp)

# Consolida todos los parciales en un resultado final por región.
chunk_result = (
    pd.concat(chunk_agg, ignore_index=True)
    .groupby('RegionGlosa', as_index=False)['NumTotal']
    .sum()
    .sort_values('NumTotal', ascending=False)
)

# Resultado de referencia: cálculo directo con dataset completo.
full_result = (
    df_raw.groupby('RegionGlosa', as_index=False)['NumTotal']
    .sum()
    .sort_values('NumTotal', ascending=False)
)

# Une ambos resultados para validar que chunking no altera cálculos.
check_chunk = chunk_result.merge(full_result, on='RegionGlosa', suffixes=('_chunk', '_full'))
# Diferencia entre método por chunk y método full.
check_chunk['Diff'] = check_chunk['NumTotal_chunk'] - check_chunk['NumTotal_full']

# Si todo está bien, la diferencia máxima debe ser 0.
print('Diferencia máxima chunk vs full:', check_chunk['Diff'].abs().max())
check_chunk.head()


## 5) Verificación de integridad y validación de esquema

In [ ]:
# Lista donde se guardan resultados de validaciones.
validation_report = []

# Helper para agregar un check al reporte con formato estándar.
def add_check(nombre: str, condicion: bool, detalle: str = ''):
    validation_report.append({
        # Nombre del control de calidad.
        'check': nombre,
        # Estado: OK si cumple, ERROR si falla.
        'resultado': 'OK' if condicion else 'ERROR',
        # Información complementaria del check.
        'detalle': detalle
    })

# Check 1: semanas en rango válido.
add_check(
    'SemanaEstadistica en [1,53]',
    df_enriched['SemanaEstadistica'].between(1, 53).all(),
    f"Fuera de rango: {(~df_enriched['SemanaEstadistica'].between(1,53)).sum()}"
)

# Conjunto permitido de categorías de triage.
valid_triage = {'C1', 'C2', 'C3', 'C4', 'C5', 'ND'}
# Check 2: triage válido.
add_check(
    'PrioridadTriage válida',
    df_enriched['PrioridadTriage'].isin(valid_triage).all(),
    f"Inválidos: {(~df_enriched['PrioridadTriage'].isin(valid_triage)).sum()}"
)

# Conjunto permitido de sexo.
valid_sexo = {'F', 'M', 'ND'}
# Check 3: sexo válido.
add_check(
    'SexoPaciente válido',
    df_enriched['SexoPaciente'].isin(valid_sexo).all(),
    f"Inválidos: {(~df_enriched['SexoPaciente'].isin(valid_sexo)).sum()}"
)

# Check 4: coherencia de totales por fila.
add_check(
    'NumTotal = suma tramos',
    (df_enriched['NumTotal'] == df_enriched[age_cols].sum(axis=1)).all(),
    'Debe ser 0 inconsistencias luego de limpieza'
)

# Regla de rango geográfico aproximado para Chile.
lat_ok = df_enriched['Latitud'].between(-56, -17)
lon_ok = df_enriched['Longitud'].between(-76, -66)
# Check 5: coordenadas plausibles.
add_check(
    'Coordenadas en rango esperado Chile',
    bool((lat_ok & lon_ok).all()),
    f"Fuera de rango: {(~(lat_ok & lon_ok)).sum()}"
)

# DataFrame final con el reporte de validación.
report_df = pd.DataFrame(validation_report)
report_df


## 6) Reporte y visualizaciones

In [ ]:
# Diccionario de KPIs para resumen ejecutivo del trabajo.
kpis = {
    # Cantidad final de filas tras limpieza.
    'registros_finales': len(df_enriched),
    # Total de atenciones acumuladas.
    'atenciones_totales': int(df_enriched['NumTotal'].sum()),
    # Costo total acumulado en CLP.
    'costo_total_clp': float(df_enriched['CostoAtencionCLP'].sum()),
    # Porcentaje de filas cuyo NumTotal fue corregido.
    'porcentaje_total_corregido': float(df_enriched['FlagTotalCorregido'].mean() * 100),
    # Causa con mayor volumen de atenciones.
    'causa_mas_frecuente': df_enriched.groupby('Causa')['NumTotal'].sum().idxmax(),
}

# Muestra KPIs calculados.
kpis


In [ ]:
# Agrega atenciones por causa y toma top 10 para gráfico.
causas_top = (
    df_enriched.groupby('Causa', as_index=False)['NumTotal']
    .sum()
    .sort_values('NumTotal', ascending=False)
    .head(10)
)

# Crea figura y tamaño.
plt.figure(figsize=(10, 5))
# Gráfico de barras horizontales causa vs atenciones.
plt.barh(causas_top['Causa'], causas_top['NumTotal'])
# Título del gráfico.
plt.title('Top 10 causas de urgencia por atenciones')
# Etiqueta eje X.
plt.xlabel('Atenciones')
# Invierte eje Y para mostrar mayor valor arriba.
plt.gca().invert_yaxis()
# Ajusta layout para evitar cortes de texto.
plt.tight_layout()
# Guarda imagen en carpeta outputs.
plt.savefig(OUTPUTS_DIR / 'top10_causas.png', dpi=120)
# Muestra gráfico en notebook.
plt.show()


In [ ]:
# Crea columna Mes (YYYY-MM) desde fecha de atención.
df_enriched['Mes'] = df_enriched['FechaAtencion'].dt.to_period('M').astype('string')
# Agrega atenciones por mes (descarta meses nulos).
serie_mensual = (
    df_enriched.dropna(subset=['Mes'])
    .groupby('Mes', as_index=False)['NumTotal']
    .sum()
)

# Crea figura de línea temporal.
plt.figure(figsize=(11, 4))
# Línea con marcador por mes.
plt.plot(serie_mensual['Mes'], serie_mensual['NumTotal'], marker='o')
# Título del gráfico.
plt.title('Evolución mensual de atenciones')
# Etiqueta eje X.
plt.xlabel('Mes')
# Etiqueta eje Y.
plt.ylabel('Atenciones')
# Rota etiquetas X para legibilidad.
plt.xticks(rotation=60)
# Ajusta layout.
plt.tight_layout()
# Guarda imagen de salida.
plt.savefig(OUTPUTS_DIR / 'evolucion_mensual_atenciones.png', dpi=120)
# Muestra gráfico.
plt.show()


In [ ]:
# Calcula costo promedio por categoría de triage.
costo_triage = (
    df_enriched.groupby('PrioridadTriage', as_index=False)['CostoAtencionCLP']
    .mean()
    .sort_values('PrioridadTriage')
)

# Crea figura de barras.
plt.figure(figsize=(7,4))
# Dibuja barras triage vs costo promedio.
plt.bar(costo_triage['PrioridadTriage'], costo_triage['CostoAtencionCLP'])
# Título del gráfico.
plt.title('Costo promedio por prioridad de triage')
# Etiqueta eje X.
plt.xlabel('Triage')
# Etiqueta eje Y.
plt.ylabel('CLP promedio')
# Ajusta layout.
plt.tight_layout()
# Guarda imagen en outputs.
plt.savefig(OUTPUTS_DIR / 'costo_promedio_triage.png', dpi=120)
# Muestra gráfico.
plt.show()


In [ ]:
# Exporta dataset procesado final (sin índice auxiliar).
df_enriched.to_csv(PROCESSED_PATH, index=False)
# Exporta reporte de validación de calidad.
report_df.to_csv(OUTPUTS_DIR / 'reporte_validacion.csv', index=False)
# Exporta KPIs en CSV para reporte/presentación.
pd.DataFrame([kpis]).to_csv(OUTPUTS_DIR / 'kpis_principales.csv', index=False)

# Calcula hash del archivo procesado para trazabilidad final.
processed_hash = sha256_file(PROCESSED_PATH)
# Muestra dónde quedó guardado.
print('Dataset procesado guardado en:', PROCESSED_PATH)
# Muestra hash del resultado final.
print('SHA256 procesado:', processed_hash)


## 7) Entorno reproducible (ítem de rúbrica)

Este notebook deja trazabilidad del entorno y crea un `requirements.txt` base.

In [ ]:
# Importa sys para mostrar versión de Python activa.
import sys
# Muestra versión de Python utilizada.
print('Python:', sys.version)
# Muestra versión de pandas del entorno.
print('pandas:', pd.__version__)
# Muestra versión de numpy del entorno.
print('numpy:', np.__version__)

# Lista de dependencias mínimas con versión fija para reproducibilidad.
requirements = [
    f"pandas=={pd.__version__}",
    f"numpy=={np.__version__}",
    f"matplotlib=={plt.matplotlib.__version__}"
]

# Escribe requirements.txt en la raíz del proyecto.
with open(PROJECT_ROOT / 'requirements.txt', 'w', encoding='utf-8') as f:
    # Une líneas con salto de línea final.
    f.write('\n'.join(requirements) + '\n')

# Confirma creación de archivo de dependencias.
print('requirements.txt generado')


## 8) Conclusiones y recomendaciones

- Se implementó un flujo **completo y reproducible**: carga, limpieza, validación, transformación y reporte.
- Se corrigieron inconsistencias críticas (duplicados, formatos, categorías y coherencia de conteos).
- Se incorporaron técnicas de escala (chunking) y eficiencia (vectorización/broadcasting).
- Recomendación: en una siguiente iteración, formalizar reglas de esquema con `pandera` y automatizar el pipeline en scripts de `src/`.